# Lab: End-to-End ETL in Microsoft Fabric Lakehouse

**Scenario:** Load sample JSON Lines retail sales data into a Fabric Lakehouse, transform the data with a Fabric Notebook, write Delta tables, handle bad data, perform a Delta MERGE/upsert, and create a Gold reporting table.

**Main flow:**

`JSON Lines file → Bronze Delta table → validation → Silver curated table + rejected rows → MERGE second batch → Gold daily summary`


## Step 0: Attach this notebook to a Lakehouse

Before running the notebook:

1. Create or open a Microsoft Fabric workspace.
2. Create a Lakehouse, for example: `lh_retail_etl_lab`.
3. Open this notebook.
4. Attach the notebook to the Lakehouse from the Lakehouse selector.
5. Run the cells in order.

This lab uses relative Lakehouse paths such as `/lakehouse/default/Files/...`, which work when the notebook is attached to a default Lakehouse.


## Step 1: Parameters and imports

This cell defines folders, file names, and table names used throughout the lab.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import os

# Lakehouse file paths
raw_folder = "/lakehouse/default/Files/retail_etl_lab/bronze"
raw_file = f"{raw_folder}/retail_sales_sample_jsonl.json"

# Lakehouse Delta table names
bronze_table = "bronze_retail_sales"
silver_table = "silver_retail_sales"
error_table = "silver_retail_sales_errors"
gold_table = "gold_daily_sales_summary"

print("Raw folder:", raw_folder)
print("Raw file:", raw_file)
print("Bronze table:", bronze_table)
print("Silver table:", silver_table)
print("Error table:", error_table)
print("Gold table:", gold_table)


## Step 2: Create sample JSON Lines data in Lakehouse Files

The source file uses **JSON Lines / NDJSON** format. Each line is one JSON object. This is easier for Spark and pipeline ingestion than a JSON array file.


In [ ]:
os.makedirs(raw_folder, exist_ok=True)

sample_jsonl = r"""{"BatchID":8001,"SourceSystem":"WEB","OrderID":4001,"OrderLineID":1,"OrderDate":"2024-03-01","CustomerID":301,"CustomerName":"James Wilson","ProductID":701,"ProductName":"Wireless Headphones","CategoryName":"Electronics","StoreID":10,"StoreName":"NYC Flagship","Quantity":2,"UnitPrice":89.99,"DiscountPct":0.00,"LoadDate":"2024-03-02"}
{"BatchID":8001,"SourceSystem":"APP","OrderID":4002,"OrderLineID":1,"OrderDate":"2024-03-02","CustomerID":302,"CustomerName":"Sophia Martinez","ProductID":702,"ProductName":"Smart Watch","CategoryName":"Electronics","StoreID":20,"StoreName":"LA Megastore","Quantity":1,"UnitPrice":249.99,"DiscountPct":10.00,"LoadDate":"2024-03-03"}
{"BatchID":8001,"SourceSystem":"POS","OrderID":4003,"OrderLineID":1,"OrderDate":"2024-03-03","CustomerID":303,"CustomerName":"Ethan Brown","ProductID":704,"ProductName":"Running Jacket","CategoryName":"Clothing","StoreID":30,"StoreName":"Chicago Downtown","Quantity":2,"UnitPrice":79.99,"DiscountPct":5.00,"LoadDate":"2024-03-04"}
{"BatchID":8001,"SourceSystem":"WEB","OrderID":4004,"OrderLineID":1,"OrderDate":"2024-03-04","CustomerID":304,"CustomerName":"Invalid Quantity Example","ProductID":703,"ProductName":"Bluetooth Speaker","CategoryName":"Electronics","StoreID":10,"StoreName":"NYC Flagship","Quantity":-1,"UnitPrice":59.99,"DiscountPct":0.00,"LoadDate":"2024-03-05"}
{"BatchID":8001,"SourceSystem":"APP","OrderID":4005,"OrderLineID":1,"OrderDate":"2024-03-05","CustomerID":305,"CustomerName":"Invalid Discount Example","ProductID":705,"ProductName":"Casual Jeans","CategoryName":"Clothing","StoreID":40,"StoreName":"Houston Mall","Quantity":1,"UnitPrice":49.99,"DiscountPct":120.00,"LoadDate":"2024-03-06"}
"""

with open(raw_file, "w", encoding="utf-8") as f:
    f.write(sample_jsonl)

print(f"Created sample JSON Lines file: {raw_file}")


## Step 3: Read the JSON Lines file into a DataFrame

This is the **Extract** step. The notebook reads raw JSON data from the Lakehouse Files area.


In [ ]:
raw_df = spark.read.json(raw_file)

display(raw_df)
raw_df.printSchema()


## Step 4: Write raw data to a Bronze Delta table

The Bronze table keeps the raw source-shaped data with minimal transformation. This is useful for replay, audit, and troubleshooting.


In [ ]:
raw_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

print(f"Bronze table created: {bronze_table}")
display(spark.table(bronze_table))


## Step 5: Validate staged/Bronze data

This is the first quality gate. Rows are labeled as valid or invalid before moving into Silver.


In [ ]:
bronze_df = spark.table(bronze_table)

validated_df = (
    bronze_df
    .withColumn(
        "ValidationStatus",
        F.when(F.col("Quantity") <= 0, F.lit("Invalid Quantity"))
         .when(F.col("UnitPrice") <= 0, F.lit("Invalid Unit Price"))
         .when((F.col("DiscountPct") < 0) | (F.col("DiscountPct") > 100), F.lit("Invalid Discount Percent"))
         .when(F.col("CustomerID") <= 0, F.lit("Invalid CustomerID"))
         .when(F.col("ProductID") <= 0, F.lit("Invalid ProductID"))
         .when(F.col("StoreID") <= 0, F.lit("Invalid StoreID"))
         .otherwise(F.lit("Valid"))
    )
)

display(validated_df.orderBy("OrderID", "OrderLineID"))


## Step 6: Split valid and invalid rows

Invalid rows go to an error table. Valid rows continue to the curated Silver table.


In [ ]:
valid_df = validated_df.filter(F.col("ValidationStatus") == "Valid")
invalid_df = (
    validated_df
    .filter(F.col("ValidationStatus") != "Valid")
    .select(
        "BatchID",
        "SourceSystem",
        "OrderID",
        "OrderLineID",
        F.col("ValidationStatus").alias("ErrorReason"),
        F.current_date().alias("ErrorCapturedDate")
    )
)

invalid_df.write.format("delta").mode("overwrite").saveAsTable(error_table)

print(f"Invalid rows written to: {error_table}")
display(spark.table(error_table))


## Step 7: Transform valid rows into Silver curated data

The Silver table adds business measures such as Gross Sales, Discount Amount, and Net Sales.


In [ ]:
silver_df = (
    valid_df
    .withColumn("GrossSalesAmount", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .withColumn("DiscountAmount", F.round(F.col("Quantity") * F.col("UnitPrice") * (F.col("DiscountPct") / 100), 2))
    .withColumn("NetSalesAmount", F.round(F.col("Quantity") * F.col("UnitPrice") * (1 - F.col("DiscountPct") / 100), 2))
    .drop("ValidationStatus")
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

print(f"Silver table created: {silver_table}")
display(spark.table(silver_table).orderBy("OrderID", "OrderLineID"))


## Step 8: Simulate a second batch and MERGE into Silver

This demonstrates an upsert:

- Order `4002` is updated with a new quantity.
- Order `4006` is inserted as a new row.


In [ ]:
second_batch_data = [
    (8002, "APP", 4002, 1, "2024-03-02", 302, "Sophia Martinez", 702, "Smart Watch", "Electronics", 20, "LA Megastore", 2, 249.99, 10.00, "2024-03-10"),
    (8002, "POS", 4006, 1, "2024-03-10", 306, "New Customer", 706, "Running Shoes", "Sports", 50, "Miami Beach", 1, 99.99, 0.00, "2024-03-11")
]

schema = StructType([
    StructField("BatchID", IntegerType(), False),
    StructField("SourceSystem", StringType(), False),
    StructField("OrderID", IntegerType(), False),
    StructField("OrderLineID", IntegerType(), False),
    StructField("OrderDate", StringType(), False),
    StructField("CustomerID", IntegerType(), False),
    StructField("CustomerName", StringType(), False),
    StructField("ProductID", IntegerType(), False),
    StructField("ProductName", StringType(), False),
    StructField("CategoryName", StringType(), False),
    StructField("StoreID", IntegerType(), False),
    StructField("StoreName", StringType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("UnitPrice", DoubleType(), False),
    StructField("DiscountPct", DoubleType(), False),
    StructField("LoadDate", StringType(), False)
])

updates_df = spark.createDataFrame(second_batch_data, schema)
updates_df = (
    updates_df
    .withColumn("OrderDate", F.to_date("OrderDate"))
    .withColumn("LoadDate", F.to_date("LoadDate"))
    .withColumn("UnitPrice", F.col("UnitPrice").cast("decimal(10,2)"))
    .withColumn("DiscountPct", F.col("DiscountPct").cast("decimal(5,2)"))
    .withColumn("GrossSalesAmount", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .withColumn("DiscountAmount", F.round(F.col("Quantity") * F.col("UnitPrice") * (F.col("DiscountPct") / 100), 2))
    .withColumn("NetSalesAmount", F.round(F.col("Quantity") * F.col("UnitPrice") * (1 - F.col("DiscountPct") / 100), 2))
)

display(updates_df)


In [ ]:
target = DeltaTable.forName(spark, silver_table)

(
    target.alias("t")
    .merge(
        updates_df.alias("s"),
        "t.OrderID = s.OrderID AND t.OrderLineID = s.OrderLineID"
    )
    .whenMatchedUpdate(set={
        "BatchID": "s.BatchID",
        "SourceSystem": "s.SourceSystem",
        "OrderDate": "s.OrderDate",
        "CustomerID": "s.CustomerID",
        "CustomerName": "s.CustomerName",
        "ProductID": "s.ProductID",
        "ProductName": "s.ProductName",
        "CategoryName": "s.CategoryName",
        "StoreID": "s.StoreID",
        "StoreName": "s.StoreName",
        "Quantity": "s.Quantity",
        "UnitPrice": "s.UnitPrice",
        "DiscountPct": "s.DiscountPct",
        "LoadDate": "s.LoadDate",
        "GrossSalesAmount": "s.GrossSalesAmount",
        "DiscountAmount": "s.DiscountAmount",
        "NetSalesAmount": "s.NetSalesAmount"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE completed.")
display(spark.table(silver_table).orderBy("OrderID", "OrderLineID"))


## Step 9: Create a Gold daily sales summary table

Gold tables are reporting-ready outputs. This table aggregates daily sales by category and store.


In [ ]:
gold_df = (
    spark.table(silver_table)
    .groupBy("OrderDate", "CategoryName", "StoreName")
    .agg(
        F.countDistinct("OrderID").alias("TotalOrders"),
        F.sum("Quantity").alias("TotalUnits"),
        F.round(F.sum("GrossSalesAmount"), 2).alias("GrossRevenue"),
        F.round(F.sum("DiscountAmount"), 2).alias("TotalDiscount"),
        F.round(F.sum("NetSalesAmount"), 2).alias("NetRevenue")
    )
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)

print(f"Gold table created: {gold_table}")
display(spark.table(gold_table).orderBy("OrderDate", "CategoryName", "StoreName"))


## Step 10: SQL checks inside the notebook

Fabric notebooks can run Spark SQL using `%%sql` cells.


In [ ]:
# Python check
print("Bronze rows:", spark.table(bronze_table).count())
print("Silver rows:", spark.table(silver_table).count())
print("Error rows:", spark.table(error_table).count())
print("Gold rows:", spark.table(gold_table).count())


In [ ]:
%%sql
SELECT CategoryName, SUM(NetSalesAmount) AS NetRevenue
FROM silver_retail_sales
GROUP BY CategoryName
ORDER BY NetRevenue DESC


## Step 11: Optional pipeline orchestration

After the notebook works, create a Fabric Data Pipeline:

1. Add a Notebook activity.
2. Select this notebook.
3. Attach the Lakehouse.
4. Run the pipeline.
5. Monitor the notebook activity output.

For a larger production pattern, use one pipeline for ingestion and one notebook for transformation, then schedule the pipeline.


## Learning summary

You completed an end-to-end Lakehouse ETL flow:

- Created sample JSON Lines source data.
- Loaded data into Bronze Delta table.
- Validated data quality.
- Stored bad rows in an error table.
- Transformed valid rows into Silver curated data.
- Used Delta MERGE for upsert logic.
- Created a Gold reporting table.
- Queried the output with Spark SQL.
